# Phi-3 · Zero-shot, few-shot y SFT

En VS Code selecciona **rocm (3.14.10)** o **Tarea 3 (GPU)**. Para instalarlo o ejecutar en Colab, sigue el [README](../README.md#entorno).

Cada configuración recibe los mismos prompts de `prompts.json` y genera con decodificación greedy y tope de 128 tokens. Zero-shot usa solo la solicitud. Few-shot antepone las cuatro demostraciones en el orden publicado. Para SFT se usa el mismo prompt de zero-shot.

Las dos primeras configuraciones pueden ejecutarse antes de entrenar. Ejecuta la última celda cuando exista el adaptador SFT correspondiente.

In [2]:
from pathlib import Path
import sys

folder = Path.cwd() if Path("comun.py").is_file() else Path.cwd() / "tarea3"
if not (folder / "comun.py").is_file():
    raise FileNotFoundError("Abre el cuaderno desde la carpeta tarea3 o la raiz del repositorio.")
sys.path.insert(0, str(folder))

from comun import preparar_notebook

preparar_notebook()

Python del cuaderno: c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Scripts\python.exe


In [3]:
import csv
import os
from time import monotonic

import torch
from peft import PeftModel
from transformers import GenerationConfig

from comun import MAX_EVAL_TOKENS, MODEL_REV, OUT, chat_tokens, load_model, load_prompts, load_tokenizer

PILOTO = os.getenv("TAREA3_PILOTO") == "1"
split = "piloto" if PILOTO else "final"
prompts = load_prompts()
cases = prompts[split]
demos = []
for item in prompts["few_shot"]:
    demos.extend([
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["respuesta"]},
    ])
print(f"Cargando Phi-3 para {len(cases)} prompts ({split})…", flush=True)
tokenizer = load_tokenizer()
model = load_model().eval()
print(f"Phi-3 listo en {torch.cuda.get_device_name(model.device)}.", flush=True)
generation = GenerationConfig(
    do_sample=False, max_new_tokens=MAX_EVAL_TOKENS,
    pad_token_id=tokenizer.pad_token_id, eos_token_id=model.generation_config.eos_token_id,
)

c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No prebuilt binary for ROCm 7.16, loading ROCm 7.15 instead. Set BNB_ROCM_VERSION to override.
W1116 21:52:07.985000 25576 .runtime\rocm\Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Cargando Phi-3 para 60 prompts (final)…


Loading weights: 100%|██████████| 195/195 [00:25<00:00,  7.79it/s]

Phi-3 listo en AMD Radeon RX 6600.


In [4]:
def generar(configuracion, model, context):
    start = monotonic()
    print(f"{configuracion} · 0/{len(cases)}", flush=True)
    results = []
    for index, row in enumerate(cases, start=1):
        messages = [*context, {"role": "user", "content": row["prompt"]}]
        ids = chat_tokens(tokenizer, messages, generation=True)
        inputs = torch.tensor([ids], device=model.device)
        with torch.inference_mode():
            output = model.generate(
                input_ids=inputs, attention_mask=torch.ones_like(inputs), generation_config=generation,
            )
        new_tokens = output[0, inputs.shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        results.append({
            "prompt_id": row["prompt_id"], "configuracion": configuracion,
            "respuesta": answer, "tokens_generados": len(new_tokens),
            "vacia": not answer, "limite_tokens": len(new_tokens) == MAX_EVAL_TOKENS,
            "revision": MODEL_REV,
        })
        if index == 1 or index % 5 == 0 or index == len(cases):
            print(f"{configuracion} · {index}/{len(cases)} · {monotonic() - start:.0f} s", flush=True)

    assert len(results) == len(cases)
    assert {row["prompt_id"] for row in results} == {row["prompt_id"] for row in cases}
    folder = OUT / "respuestas" / split
    folder.mkdir(parents=True, exist_ok=True)
    file = folder / f"{configuracion}.csv"
    temp = file.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    temp.replace(file)
    print(f"{file.name} · {len(results)} respuestas · {sum(row['vacia'] for row in results)} vacías")

In [5]:
generar("phi3_zero_shot", model, [])

phi3_zero_shot · 0/60
phi3_zero_shot · 1/60 · 6 s
phi3_zero_shot · 5/60 · 70 s
phi3_zero_shot · 10/60 · 114 s
phi3_zero_shot · 15/60 · 167 s
phi3_zero_shot · 20/60 · 194 s
phi3_zero_shot · 25/60 · 231 s
phi3_zero_shot · 30/60 · 288 s
phi3_zero_shot · 35/60 · 465 s
phi3_zero_shot · 40/60 · 641 s
phi3_zero_shot · 45/60 · 817 s
phi3_zero_shot · 50/60 · 986 s
phi3_zero_shot · 55/60 · 1150 s
phi3_zero_shot · 60/60 · 1326 s
phi3_zero_shot.csv · 60 respuestas · 0 vacías


In [6]:
generar("phi3_few_shot", model, demos)

phi3_few_shot · 0/60
phi3_few_shot · 1/60 · 11 s
phi3_few_shot · 5/60 · 62 s
phi3_few_shot · 10/60 · 127 s
phi3_few_shot · 15/60 · 188 s
phi3_few_shot · 20/60 · 246 s
phi3_few_shot · 25/60 · 307 s
phi3_few_shot · 30/60 · 363 s
phi3_few_shot · 35/60 · 464 s
phi3_few_shot · 40/60 · 559 s
phi3_few_shot · 45/60 · 626 s
phi3_few_shot · 50/60 · 703 s
phi3_few_shot · 55/60 · 775 s
phi3_few_shot · 60/60 · 863 s
phi3_few_shot.csv · 60 respuestas · 0 vacías


In [7]:
adapter = OUT / ("sft_piloto" if PILOTO else "sft")
if not (adapter / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Ejecuta el entrenamiento SFT para obtener {adapter}")
sft_model = PeftModel.from_pretrained(model, str(adapter)).eval()
generar("phi3_sft", sft_model, [])

phi3_sft · 0/60


c:\Users\diego\Documents\GitHub\DeepLearning\tarea3\.runtime\rocm\Lib\site-packages\transformers\modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


phi3_sft · 1/60 · 4 s
phi3_sft · 5/60 · 24 s
phi3_sft · 10/60 · 47 s
phi3_sft · 15/60 · 67 s
phi3_sft · 20/60 · 86 s
phi3_sft · 25/60 · 107 s
phi3_sft · 30/60 · 126 s
phi3_sft · 35/60 · 184 s
phi3_sft · 40/60 · 243 s
phi3_sft · 45/60 · 279 s
phi3_sft · 50/60 · 335 s
phi3_sft · 55/60 · 384 s
phi3_sft · 60/60 · 451 s
phi3_sft.csv · 60 respuestas · 0 vacías
